<a href="https://colab.research.google.com/github/vivekgautamgv/Python-For-Finance/blob/main/renko_chart_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ccxt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.0/131.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.3/289.3 kB 24.8 MB/s eta 0:00:00
  Attempting uninstall: aiohttp
    Found existing installation: aiohttp 3.11.15
    Uninstalling aiohttp-3.11.15:
      Successfully uninstalled aiohttp-3.11.15


In [ ]:
!pip install pandas_ta

In [ ]:
! pip install pandas-ta

In [ ]:
import ccxt
from datetime import datetime
import pandas as pd
import scipy.optimize as opt
import numpy as np
import pandas_ta as ta
from stocktrends import Renko
import mplfinance as mpf
import matplotlib.pyplot as plt

ImportError: cannot import name 'NaN' from 'numpy' (/usr/local/lib/python3.11/dist-packages/numpy/__init__.py)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime, timedelta

# Define parameters
BRICK_SIZE = 1.7
STOCK_SYMBOL = "ZOMATO.NS"  # Yahoo Finance symbol for Zomato/Eternal Ltd
START_DATE = datetime.now() - timedelta(days=90)  # 3 months ago
END_DATE = datetime.now()
TIME_FRAME = "5m"  # 5-minute data

def download_data(symbol, start_date, end_date, interval):
    """Download stock data from Yahoo Finance."""
    print(f"Downloading data for {symbol} from {start_date} to {end_date}...")
    data = yf.download(symbol, start=start_date, end=end_date, interval=interval)
    return data

def create_renko_chart(df, brick_size):
    """Convert OHLC data to Renko chart data."""
    renko_data = []
    if df.empty:
        return pd.DataFrame()

    # Initialize with the first closing price
    current_price = df['Close'].iloc[0]
    current_direction = 0  # 0: neutral, 1: up, -1: down

    for idx, row in df.iterrows():
        close_price = row['Close']

        # Calculate price difference and number of bricks
        price_diff = close_price - current_price
        bricks = int(abs(price_diff) / brick_size)

        if bricks == 0:
            continue

        # Determine direction of the bricks
        direction = 1 if price_diff > 0 else -1

        # Add bricks to the renko chart
        for _ in range(bricks):
            current_price += direction * brick_size
            renko_data.append({
                'datetime': idx,
                'open': current_price - direction * brick_size,
                'high': max(current_price, current_price - direction * brick_size),
                'low': min(current_price, current_price - direction * brick_size),
                'close': current_price,
                'direction': direction
            })

    return pd.DataFrame(renko_data)

def backtest_strategy(renko_df):
    """Backtest the 3 consecutive bricks strategy."""
    if len(renko_df) < 4:
        return pd.DataFrame()

    # Initialize trade signals
    renko_df['signal'] = 0
    trades = []

    # Look for 3 consecutive bricks of the same direction
    for i in range(3, len(renko_df)):
        # Check if last 3 bricks are in the same direction
        if (renko_df['direction'].iloc[i-3] == renko_df['direction'].iloc[i-2] == renko_df['direction'].iloc[i-1]):
            # Take position in the same direction
            position_direction = renko_df['direction'].iloc[i-1]
            entry_price = renko_df['open'].iloc[i]
            exit_price = renko_df['close'].iloc[i]
            profit = position_direction * (exit_price - entry_price)

            trade = {
                'entry_time': renko_df['datetime'].iloc[i],
                'exit_time': renko_df['datetime'].iloc[i],
                'entry_price': entry_price,
                'exit_price': exit_price,
                'direction': 'LONG' if position_direction == 1 else 'SHORT',
                'profit': profit,
                'profit_pct': profit / entry_price * 100
            }
            trades.append(trade)
            renko_df.loc[renko_df.index[i], 'signal'] = position_direction

    trades_df = pd.DataFrame(trades)
    return trades_df, renko_df

def analyze_performance(trades_df):
    """Calculate performance metrics for the strategy."""
    if trades_df.empty:
        return "No trades were executed."

    # Basic statistics
    total_trades = len(trades_df)
    winning_trades = len(trades_df[trades_df['profit'] > 0])
    losing_trades = len(trades_df[trades_df['profit'] <= 0])
    win_rate = winning_trades / total_trades * 100 if total_trades > 0 else 0

    # Long vs Short trades
    long_trades = len(trades_df[trades_df['direction'] == 'LONG'])
    short_trades = len(trades_df[trades_df['direction'] == 'SHORT'])

    # Profit metrics
    total_profit = trades_df['profit'].sum()
    average_profit = trades_df['profit'].mean()
    max_profit = trades_df['profit'].max()
    max_loss = trades_df['profit'].min()

    # Risk metrics
    profit_factor = abs(trades_df[trades_df['profit'] > 0]['profit'].sum() /
                       trades_df[trades_df['profit'] < 0]['profit'].sum()) if trades_df[trades_df['profit'] < 0]['profit'].sum() != 0 else float('inf')

    # Create summary
    summary = {
        'Total Trades': total_trades,
        'Winning Trades': winning_trades,
        'Losing Trades': losing_trades,
        'Win Rate (%)': round(win_rate, 2),
        'Long Trades': long_trades,
        'Short Trades': short_trades,
        'Total Profit': round(total_profit, 2),
        'Average Profit per Trade': round(average_profit, 2),
        'Maximum Profit': round(max_profit, 2),
        'Maximum Loss': round(max_loss, 2),
        'Profit Factor': round(profit_factor, 2),
    }

    return summary

def plot_renko_with_signals(renko_df):
    """Plot Renko chart with trade signals."""
    if renko_df.empty:
        return

    plt.figure(figsize=(14, 7))

    # Plot Renko bricks
    for i in range(len(renko_df)):
        color = 'green' if renko_df['direction'].iloc[i] > 0 else 'red'
        signal_color = 'blue' if renko_df['signal'].iloc[i] != 0 else color

        plt.plot([i, i+1], [renko_df['open'].iloc[i], renko_df['open'].iloc[i]], color=signal_color, linewidth=2)
        plt.plot([i, i+1], [renko_df['close'].iloc[i], renko_df['close'].iloc[i]], color=signal_color, linewidth=2)
        plt.plot([i, i], [renko_df['open'].iloc[i], renko_df['close'].iloc[i]], color=signal_color, linewidth=2)
        plt.plot([i+1, i+1], [renko_df['open'].iloc[i], renko_df['close'].iloc[i]], color=signal_color, linewidth=2)

        # Highlight trades
        if renko_df['signal'].iloc[i] != 0:
            entry_text = 'BUY' if renko_df['signal'].iloc[i] > 0 else 'SELL'
            plt.text(i+0.5, renko_df['close'].iloc[i] + 0.5, entry_text,
                     horizontalalignment='center', color='blue', fontweight='bold')

    plt.title('Renko Chart with Trade Signals (Brick Size: {} INR)'.format(BRICK_SIZE))
    plt.xlabel('Brick Number')
    plt.ylabel('Price (INR)')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('renko_chart_with_signals.png')
    plt.close()

def main():
    # Download data
    data = download_data(STOCK_SYMBOL, START_DATE, END_DATE, TIME_FRAME)

    if data.empty:
        print("No data available for the specified symbol and time period.")
        return

    # Create Renko chart
    renko_df = create_renko_chart(data, BRICK_SIZE)
    print(f"Total number of Renko bricks formed: {len(renko_df)}")

    if renko_df.empty:
        print("Could not create Renko chart. Not enough price movement.")
        return

    # Backtest strategy
    trades_df, renko_df_with_signals = backtest_strategy(renko_df)

    # Analyze performance
    performance = analyze_performance(trades_df)
    print("\n--- STRATEGY PERFORMANCE ---")
    for key, value in performance.items():
        print(f"{key}: {value}")

    # Plot Renko chart with signals
    plot_renko_with_signals(renko_df_with_signals)
    print("\nRenko chart with signals has been saved as 'renko_chart_with_signals.png'")

if __name__ == "__main__":
    main()

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ZOMATO.NS']: YFPricesMissingError('possibly delisted; no price data found  (5m 2025-01-14 06:06:04.544432 -> 2025-04-14 06:06:04.544502) (Yahoo error = "5m data not available for startTime=1736814964 and endTime=1744590964. The requested range must be within the last 60 days.")')


No data available for the specified symbol and time period.
